# Lesson 13 — Long-Term Memory with Vector Stores

## What you will learn
- The **3 memory types** in AI agents and when to use each
- How to **extract facts** from conversations automatically
- How to store facts in a **per-user vector store**
- How to retrieve **relevant memories** at the start of each turn
- **Cross-thread memory**: global user profile that persists between sessions

## Memory Hierarchy
```
┌─────────────────┬───────────────────────────────┬──────────────────────┐
│ Type            │ Where                         │ Best for             │
├─────────────────┼───────────────────────────────┼──────────────────────┤
│ In-context      │ Full messages list in prompt   │ Short conversations  │
│ Structured DB   │ SQLite/Postgres key-value      │ Exact fact lookup    │
│ Vector store    │ Chroma / semantic embeddings   │ "What did user say?" │
└─────────────────┴───────────────────────────────┴──────────────────────┘
```

## Per-turn pattern
```
1. Load memories  → search vector store for relevant past facts
2. Chat           → include loaded memories as extra context
3. Save memories  → extract new facts → embed → store
```

In [ ]:
# !pip install langgraph langchain-ollama langchain-community chromadb

## Step 1 — Setup

In [ ]:
import json
import os
from typing import Annotated, Optional
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

try:
    from langchain_community.vectorstores import Chroma
    CHROMA_AVAILABLE = True
    print("✅ Chroma available")
except ImportError:
    CHROMA_AVAILABLE = False
    print("⚠️  Install: pip install langchain-community chromadb")

llm = ChatOllama(model="llama3", temperature=0.3)

# Per-user vector stores (keyed by user_id)
_user_stores: dict = {}

## Step 2 — Memory Store Helpers

In [ ]:
def get_user_store(user_id: str):
    """Get or create a per-user Chroma vector store."""
    if not CHROMA_AVAILABLE:
        return None
    if user_id not in _user_stores:
        embeddings = OllamaEmbeddings(model="nomic-embed-text")
        _user_stores[user_id] = Chroma(
            collection_name=f"memory_{user_id}",
            embedding_function=embeddings,
            persist_directory=f"./chroma_memory_{user_id}",
        )
    return _user_stores[user_id]

def load_memories(user_id: str, query: str, k: int = 3) -> str:
    """Retrieve top-k relevant memories for this user."""
    store = get_user_store(user_id)
    if store is None:
        return ""
    try:
        results = store.similarity_search(query, k=k)
        if not results:
            return ""
        facts = "\n".join([f"- {r.page_content}" for r in results])
        return f"Relevant memories for this user:\n{facts}"
    except Exception:
        return ""

def save_memory(user_id: str, fact: str):
    """Store a new fact in the user's memory."""
    store = get_user_store(user_id)
    if store is None:
        return
    store.add_documents([Document(page_content=fact)])
    print(f"  [memory] Saved for {user_id}: '{fact[:60]}'")

print("Memory helpers ready")

## Step 3 — State Definition

In [ ]:
class MemoryState(TypedDict):
    messages:       Annotated[list, add_messages]
    user_id:        str
    loaded_memory:  str   # injected from vector store each turn
    facts_to_save:  list  # extracted facts to persist after turn

## Step 4 — Graph Nodes

In [ ]:
def load_memory_node(state: MemoryState) -> dict:
    """Load relevant memories before calling the LLM."""
    last_msg = state["messages"][-1].content if state["messages"] else ""
    memories = load_memories(state["user_id"], last_msg)
    return {"loaded_memory": memories}

def chat_node(state: MemoryState) -> dict:
    """Call LLM with memory-augmented context."""
    system_parts = ["You are a helpful personalized assistant with long-term memory."]
    if state["loaded_memory"]:
        system_parts.append(state["loaded_memory"])
    system_parts.append("Use this context naturally in your response.")
    system = SystemMessage(content="\n\n".join(system_parts))
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

def extract_memory_node(state: MemoryState) -> dict:
    """Extract saveable facts from the latest exchange."""
    last_human = next(
        (m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), ""
    )
    prompt = [
        SystemMessage(content=(
            "Extract 1-2 specific, reusable facts about the user from their message. "
            "Return JSON: {\"facts\": [\"fact1\", \"fact2\"]}. "
            "If nothing memorable, return {\"facts\": []}."
        )),
        HumanMessage(content=f"User said: {last_human}"),
    ]
    resp = llm.invoke(prompt)
    try:
        data = json.loads(resp.content)
        facts = data.get("facts", [])
    except Exception:
        facts = []
    # Save immediately
    for fact in facts:
        save_memory(state["user_id"], fact)
    return {"facts_to_save": facts}

## Step 5 — Build the Graph
```
START → load_memory → chat → extract_memory → END
```

In [ ]:
checkpointer = MemorySaver()

builder = StateGraph(MemoryState)
builder.add_node("load_memory",    load_memory_node)
builder.add_node("chat",           chat_node)
builder.add_node("extract_memory", extract_memory_node)

builder.add_edge(START,            "load_memory")
builder.add_edge("load_memory",    "chat")
builder.add_edge("chat",           "extract_memory")
builder.add_edge("extract_memory", END)

graph = builder.compile(checkpointer=checkpointer)
print("Memory graph compiled!")

## Step 6 — Run Multi-Turn Conversation with Persistent Memory

In [ ]:
def chat(user_id: str, message: str):
    config = {"configurable": {"thread_id": user_id}}
    result = graph.invoke(
        {"messages": [HumanMessage(content=message)], "user_id": user_id,
         "loaded_memory": "", "facts_to_save": []},
        config=config,
    )
    reply = result["messages"][-1].content
    return reply, result.get("facts_to_save", [])

# Conversation with Alice
conversations = [
    ("alice", "Hi! My name is Alice and I'm a Python developer who loves LangGraph."),
    ("alice", "I prefer concise answers with code examples."),
    ("alice", "What are the best practices for LangGraph in production?"),
    # New session — memories should be recalled
    ("alice", "Can you remind me what you know about me?"),
]

for user_id, msg in conversations:
    print(f"\n[{user_id}] You: {msg[:70]}")
    reply, facts = chat(user_id, msg)
    print(f"[{user_id}] Bot: {reply[:200]}")
    if facts:
        print(f"  💾 Saved: {facts}")

## Key Takeaways

| Concept | Detail |
|---------|--------|
| `load_memory_node` | Runs BEFORE LLM — injects relevant past facts |
| `extract_memory_node` | Runs AFTER LLM — saves new facts to vector store |
| Per-user `Chroma` collection | Isolated memory per user — no cross-contamination |
| `similarity_search(query, k=3)` | Retrieves semantically relevant (not exact) memories |
| Fact extraction via LLM | Converts free-form conversation → structured memories |

## 🏋️ Exercise
1. Add a `forget_memory(user_id, keyword)` function that deletes memories containing a keyword
2. Add a tool `show_my_memories` the agent can call to list all stored facts for the user
3. Test: tell the bot your name and preferences, restart the notebook, start a new conversation — do memories persist?

In [ ]:
# Your exercise solution here
